In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =========================
# Configuration
# =========================

BRONZE_TABLE = "dbx_fintech_data_platform.bronze.customers"
SILVER_TABLE = "dbx_fintech_data_platform.silver.customers"

In [0]:
silver_source_df = spark.table(BRONZE_TABLE)
display(silver_source_df)

In [0]:
silver_source_df.printSchema()

In [0]:
null_blank_check = silver_source_df.select(
    *[
        F.sum(
            F.when(
                F.col(c).isNull() |
                (F.trim(F.col(c).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in silver_source_df.columns
    ]
)

display(null_blank_check)

In [0]:
silver_clean_df = (
    silver_source_df
    .withColumn("customer_id",F.trim(F.col("customer_id")))
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("country", F.upper(F.trim(F.col("country"))))
    .withColumn("customer_type", F.upper(F.trim(F.col("customer_type"))))
)
display(silver_clean_df)

In [0]:
silver_clean_df = silver_clean_df.withColumn(
    "phone", F.col("phone").cast("string")
)
display(silver_clean_df)

In [0]:
display(
    silver_clean_df
    .groupBy("country")
    .count()
    .orderBy("country")
)

display(
    silver_clean_df
    .groupBy("customer_type")
    .count()
    .orderBy("customer_type")
)

In [0]:
valid_countries = ["CA", "DE", "IN", "UK", "US"]
valid_customer_types = ["STANDARD", "PREMIUM", "VIP"]

invalid_df = silver_clean_df.filter(
    (~F.col("country").isin(valid_countries)) |
    (~F.col("customer_type").isin(valid_customer_types))
)

display(invalid_df)

In [0]:
customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("_source_date").desc(),
        F.col("updated_at").desc()
    )
    )

ranked_df = (
    silver_clean_df.withColumn("_row_number", F.row_number().over(customer_window))
)
display(
    ranked_df
    .select(
        "customer_id",
        "updated_at",
        "_source_date",
        "_row_number"
    )
    .orderBy("customer_id")
)


In [0]:

silver_current_df = ranked_df.filter(F.col("_row_number")==1).drop("_row_number")
display(silver_current_df.count())


In [0]:
print("Total records:", silver_current_df.count())

print(
    "Unique customers:",
    silver_current_df.select("customer_id").distinct().count()
)

In [0]:
silver_final_df = silver_current_df.select(
    "customer_id",
    "name",
    "email",
    "phone",
    "country",
    "customer_type",
    "created_at",
    "updated_at",
    "_ingestion_timestamp",
    "_source_file",
    "_source_date"
)

silver_final_df.printSchema()

In [0]:
duplicate_count = (
    silver_final_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate customer IDs:", duplicate_count)

In [0]:
mandatory_columns = [
    "customer_id",
    "email",
    "phone",
    "country",
    "customer_type"
]

mandatory_null_check = silver_final_df.select(
    *[
        F.sum(
            F.when(
                F.col(c).isNull() |
                (F.trim(F.col(c).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in mandatory_columns
    ]
)

display(mandatory_null_check)

In [0]:
print("Silver customer count:", silver_final_df.count())

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS dbx_fintech_data_platform.silver
""")

In [0]:
(
    silver_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

In [0]:
spark.sql(f"""
DESCRIBE TABLE {SILVER_TABLE}
""").show(truncate=False)

In [0]:
display(
    spark.table(SILVER_TABLE)
)

In [0]:
print(
    "Silver records:",
    spark.table(SILVER_TABLE).count()
)

print(
    "Unique customers:",
    spark.table(SILVER_TABLE)
    .select("customer_id")
    .distinct()
    .count()
)

In [0]:
test_data = [
    ("C001", " Alice ", "ALICE@EMAIL.COM", "9876543210", "in", "premium"),
    ("C002", "Bob", "", "9876543211", "US", "STANDARD"),
    ("C003", "Charlie", "charlie@email.com", "9876543212", "XX", "VIP"),
    ("C004", "David", "david@email.com", "9876543213", "UK", "UNKNOWN"),
    ("C005", "Eve", "eve@email.com", "9876543214", "IN", "STANDARD"),
    ("C005", "Eve", "eve@email.com", "9876543214", "IN", "STANDARD"),
]

test_columns = [
    "customer_id",
    "name",
    "email",
    "phone",
    "country",
    "customer_type"
]

test_df = spark.createDataFrame(test_data, test_columns)

display(test_df)

In [0]:
test_clean_df = (
    test_df
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("country", F.upper(F.trim(F.col("country"))))
    .withColumn("customer_type", F.upper(F.trim(F.col("customer_type"))))
    .withColumn("phone", F.col("phone").cast("string"))
)

display(test_clean_df)

In [0]:
invalid_test_df = test_clean_df.filter(
    (~F.col("country").isin(valid_countries)) |
    (~F.col("customer_type").isin(valid_customer_types))
)

display(invalid_test_df)

In [0]:
mandatory_test_df = test_clean_df.filter(
    F.col("customer_id").isNull() |
    (F.trim(F.col("customer_id")) == "") |
    F.col("email").isNull() |
    (F.trim(F.col("email")) == "") |
    F.col("phone").isNull() |
    (F.trim(F.col("phone")) == "") |
    F.col("country").isNull() |
    (F.trim(F.col("country")) == "") |
    F.col("customer_type").isNull() |
    (F.trim(F.col("customer_type")) == "")
)

display(mandatory_test_df)

In [0]:
duplicate_test_df = (
    test_clean_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_test_df)

In [0]:
from datetime import datetime

duplicate_test_data = [
    ("C005", "Eve", "eve@email.com", "9876543214", "IN", "STANDARD",
     datetime(2026, 8, 17, 10, 0, 0)),
     
    ("C005", "Eve Updated", "eve@email.com", "9876543214", "IN", "STANDARD",
     datetime(2026, 8, 18, 10, 0, 0)),
]

duplicate_test_columns = [
    "customer_id",
    "name",
    "email",
    "phone",
    "country",
    "customer_type",
    "updated_at"
]

duplicate_test_df = spark.createDataFrame(
    duplicate_test_data,
    duplicate_test_columns
)

display(duplicate_test_df)

In [0]:
duplicate_ranked_df = duplicate_test_df.withColumn(
    "rank",
    F.row_number().over(
        Window
        .partitionBy("customer_id")
        .orderBy(F.col("updated_at").desc())
    )
)

display(duplicate_ranked_df)